# Config – Metadata Driven JSON Loader


Two tables drive this pipeline:
1. **source_control** – the sources to process (source, file_path,
   file_type, target catalog/schema/table, is_active)
2. **load_logs** – every run and every file attempt gets a row here. On a
   re-run, this table is checked to work out which files already
   succeeded and which still need loading

## Required Libraries

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, TimestampType
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import uuid
import re

In [0]:
%sql
select * from iics.casemix.json_test limit 1

In [0]:
%sql
describe iics.casemix.json_test

## Meta table locations

In [0]:
meta_catalog = "iics"
meta_schema  = "casemix"

source_control_table = "json_control_test"   # source config
log_table             = "json_logs_test"        # audit log

#executing user
executed_by = spark.sql("SELECT current_user()").first()[0]

In [0]:
source_control_table = "json_control_test"

spark.sql(f"DROP TABLE IF EXISTS iics.casemix.{source_control_table}")

    spark.sql(f"""
    CREATE TABLE iics.casemix.{source_control_table} (
        s_no                       INT,
        source                     STRING,
        file_path                  STRING,
        file_type                  STRING,
        tgt_catalog                STRING,
        tgt_schema                 STRING,
        tgt_table                  STRING,
        tgt_checkpoint_path        STRING,
        versioning_partition_field STRING,
        is_full_load               INT,
        is_incremental_load        INT,
        is_active                  INT
    )
    USING DELTA
""")

spark.sql(f"""
    INSERT INTO iics.casemix.{source_control_table}
    VALUES
        (1, 's3', '/Volumes/iics/casemix/history/casemix/zip_files/processed/', 'json',
         'iics', 'casemix', 'json_test',
         '/Volumes/iics/casemix/history/casemix/zip_files/processed/check_point',
         'file_name', 0, 0, 0),

        (2, 's3', '/Volumes/iics/casemix/history/json_stress_test_1_gb/', 'json',
         'iics', 'casemix', 'json_stress_test_1_gb',
         '/Volumes/iics/casemix/history/json_stress_test_1_gb_checkpoint/',
         'file_name', 0, 0, 0),

        (3, 's3', '/Volumes/iics/casemix/history/json_stress_test_1/', 'json',
         'iics', 'casemix', 'json_stress_test_1',
         '/Volumes/iics/casemix/history/json_stress_test_1_checkpoint/',
         'file_name', 0, 0, 0),

        (4, 's3', '/Volumes/iics/casemix/history/json_test_10_files/', 'json',
         'iics', 'casemix', 'json_test_10_files',
         '/Volumes/iics/casemix/history/json_test_10_files_checkpoint/',
         'file_name', 0, 0, 0),

        (5, 's3', '/Volumes/iics/casemix/history/json_stress_test/', 'json',
         'iics', 'casemix', 'json_stress_test',
         '/Volumes/iics/casemix/history/json_stress_test_checkpoint/',
         'file_name', 0, 0, 0)
""")

print(f"Seeded iics.casemix.{source_control_table} with default rows")

In [0]:
def create_log_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {meta_catalog}.{meta_schema}.{log_table} (
            log_id          STRING,
            run_id          STRING,
            executed_by     STRING,
            s_no            INT,
            source          STRING,
            file_path       STRING,
            tgt_catalog     STRING,
            tgt_schema      STRING,
            tgt_table       STRING,
            file_name       STRING,
            status          STRING,
            start_time      TIMESTAMP,
            end_time        TIMESTAMP,
            rows_processed  BIGINT,
            message         STRING
        ) USING DELTA
    """)


LOG_SCHEMA = StructType([
    StructField("log_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("executed_by", StringType(), True),
    StructField("s_no", IntegerType(), True),
    StructField("source", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("tgt_catalog", StringType(), True),
    StructField("tgt_schema", StringType(), True),
    StructField("tgt_table", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("message", StringType(), True),
])

## Run identifiers

Every log row from one execution shares the same `run_id`.

In [0]:
notebook_name = "02_main"
run_id = str(uuid.uuid4())